In [1]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client

# Project root directory
BASE_DIR = Path.cwd().parent

# Load environment variables
load_dotenv(BASE_DIR / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

# Create Supabase connection
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase connection initialized successfully!")

Supabase connection initialized successfully!


In [3]:
TABLES = {
    "customers": "customer_id",
    "merchants": "merchant_id",
    "devices": "device_id",
    "transactions": "transaction_id",
    "payment_outcomes": "transaction_id",
    "labels": "transaction_id",
    "features": "transaction_id"
}

def load_table(table_name, primary_key):
    all_rows = []
    start = 0
    batch_size = 1000

    while True:
        response = (
            supabase.table(table_name)
            .select("*")
            .order(primary_key)
            .range(start, start + batch_size - 1)
            .execute()
        )

        rows = response.data or []

        if not rows:
            break

        all_rows.extend(rows)
        start += len(rows)

        if len(rows) < batch_size:
            break

    return pd.DataFrame(all_rows)


# Load all tables into Pandas
dataframes = {}

for table_name, primary_key in TABLES.items():
    print(f"Loading {table_name}...")

    dataframes[table_name] = load_table(
        table_name,
        primary_key
    )

    print(f"Loaded: {len(dataframes[table_name]):,} rows")


print("\nAll tables loaded successfully!")

Loading customers...
Loaded: 2,500 rows
Loading merchants...
Loaded: 200 rows
Loading devices...
Loaded: 10,120 rows
Loading transactions...
Loaded: 110,000 rows
Loading payment_outcomes...
Loaded: 110,000 rows
Loading labels...
Loaded: 110,000 rows
Loading features...
Loaded: 110,000 rows

All tables loaded successfully!


In [4]:
for table_name, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print(f"Shape: {df.shape}")

    display(df.head(3))


TABLE: customers
Shape: (2500, 6)


,customer_id,created_at,customer_age,home_city,region,is_synthetic
0,C_0004ff2f7739f0d3,2023-04-02T18:30:00+00:00,60,Jaipur,north,True
1,C_00051dee81a46e47,2022-08-01T18:30:00+00:00,39,Lucknow,north,True
2,C_001ba2fd8bb7865d,2023-01-07T18:30:00+00:00,58,Kochi,south,True



TABLE: merchants
Shape: (200, 5)


,merchant_id,synthetic_name,merchant_category,merchant_location,is_synthetic
0,M_00208f2ea071efdc,Synthetic Merchant 000083,travel,"{'lat': 18.52, 'lon': 73.86, 'city': 'Pune'}",True
1,M_00a65588a18e5cb2,Synthetic Merchant 000177,quick_commerce,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}",True
2,M_04c9599428747d92,Synthetic Merchant 000062,education,"{'lat': 22.72, 'lon': 75.86, 'city': 'Indore'}",True



TABLE: devices
Shape: (10120, 3)


,device_id,device_kind,is_synthetic
0,D_000ce3ddcb56a49d,mobile,True
1,D_00135d58adae46a7,mobile,True
2,D_001e2d8c7b1ca90e,mobile,True


... (4 outputs left out)


TABLE: labels
Shape: (110000, 7)


,transaction_id,fraud_label,label_source,label_available_at,split,eligible_at_next_split_start,is_synthetic
0,TX_0000000001,0,mature_no_fraud,2024-11-29T18:35:15.602+00:00,warmup,True,True
1,TX_0000000002,0,mature_no_fraud,2024-11-29T18:36:10.733+00:00,warmup,True,True
2,TX_0000000003,0,mature_no_fraud,2024-11-29T18:37:47.985+00:00,warmup,True,True



TABLE: features
Shape: (110000, 22)


,transaction_id,feature_as_of,amount_to_avg_ratio,is_new_device,customer_merchant_distance_km,is_new_city,hour_of_day,is_night,has_recent_failures,is_new_account,...,distinct_customers_per_device_7d,combined_risk_flags,mcc_target_encoded_rate,mcc_encoding_as_of,mcc_encoding_count,missing_customer_location,missing_device,cold_start,reason_codes,feature_version
0,TX_0000000001,2024-09-30T18:35:15.602+00:00,NaN,True,NaN,None,0,True,False,False,...,0.0,2,0.006,2024-09-30T18:30:00+00:00,0,True,False,True,"[R01_NEW_DEVICE, R03_ODD_HOUR, R08_HIGH_RISK_MCC]",sentinel-features-1.0.0
1,TX_0000000002,2024-09-30T18:36:10.733+00:00,2.9350,False,845.38,True,0,True,False,False,...,1.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,False,"[R02_GEO_MISMATCH, R03_ODD_HOUR, R08_HIGH_RISK...",sentinel-features-1.0.0
2,TX_0000000003,2024-09-30T18:37:47.985+00:00,0.7285,False,845.38,False,0,True,False,False,...,1.0,2,0.006,2024-09-30T18:30:00+00:00,0,False,False,False,"[R02_GEO_MISMATCH, R03_ODD_HOUR, R08_HIGH_RISK...",sentinel-features-1.0.0


In [5]:
customers_df = dataframes["customers"]
merchants_df = dataframes["merchants"]
devices_df = dataframes["devices"]
transactions_df = dataframes["transactions"]
payment_outcomes_df = dataframes["payment_outcomes"]
labels_df = dataframes["labels"]
features_df = dataframes["features"]

In [6]:
# Export all seven DataFrames to CSV

DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

for table_name, df in dataframes.items():
    file_path = DATA_DIR / f"{table_name}.csv"

    df.to_csv(file_path, index=False)

    print(f"Saved {table_name}: {len(df):,} rows")

Saved customers: 2,500 rows
Saved merchants: 200 rows
Saved devices: 10,120 rows
Saved transactions: 110,000 rows
Saved payment_outcomes: 110,000 rows
Saved labels: 110,000 rows
Saved features: 110,000 rows
